# Model Registry: Registration, Versioning & Promotion

**Session 2 -- Deep Dive: Feature Engineering, Feature Stores & Experiment Tracking**  
**Notebook 4 of 4** | ~15 minutes

---

## What You Will Learn

| Concept | What We Cover |
|---------|---------------|
| **Model Registration** | Log trained models into the Snowflake Model Registry |
| **Versioning** | Create and manage multiple model versions |
| **Metadata & Metrics** | Attach metrics, comments, and metadata to versions |
| **Model Lineage** | Trace upstream from model to dataset to feature view |
| **Version Comparison** | Compare metrics across model versions side-by-side |
| **Promotion Workflow** | Evaluate against thresholds and decide on promotion |

---

## Snowflake Model Registry Capabilities

| Capability | How It Works |
|------------|-------------|
| Registration | `registry.log_model()` — single call to serialize, store, and capture metadata |
| Versioning | Named versions (e.g. `v1`, `v2`) — human-readable and supports semantic versioning |
| Promotion | Default version abstraction + custom threshold logic |
| Model serving | Warehouse inference (SQL) or Snowpark Container Services (SPCS) |
| Lineage | Automatic: traces from Dataset through Feature View to source table |
| Governance | Snowflake RBAC — same permission model as your data |
| Explainability | Built-in `explain()` method for model interpretability |

### Prerequisites

Run notebooks 01-03 first. This notebook uses the best model and dataset from those notebooks.

## 1 | Environment Setup

In [ ]:
%cd ..
%load_ext autoreload

In [ ]:
from datetime import datetime
import logging

import pandas as pd
from snowflake.ml.modeling.linear_model import LogisticRegression

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

from utils import get_config
from utils import get_feature_config, get_session

config = get_config("config.yaml")
session = get_session(config.snowflake.connection_name)

DB = config.snowflake.database
SCHEMA = config.snowflake.schema_name
WH = config.snowflake.warehouse

session.use_database(DB)
session.use_schema(SCHEMA)
session.use_warehouse(WH)

print(f"Connected: {DB}.{SCHEMA}")

## 2 | Train Model v1 with Snowflake ML LogisticRegression

We use `snowflake.ml.modeling.linear_model.LogisticRegression` which works directly
with Snowpark DataFrames, handles string labels natively, and trains quickly.
We use only numeric features to keep things simple.

In [ ]:
from snowflake.ml.dataset import load_dataset
from snowflake.ml.modeling.metrics import accuracy_score, f1_score, precision_score, recall_score

ds_name = "PATIENT_RISK_TRAINING_DS"

versions = session.sql(f"SHOW VERSIONS IN DATASET {DB}.{SCHEMA}.{ds_name}").collect()
ds_version = versions[-1]["version"]

dataset = load_dataset(session, f"{DB}.{SCHEMA}.{ds_name}", version=ds_version)
train_spdf = dataset.read.to_snowpark_dataframe()

test_spdf = session.table(f"{DB}.{SCHEMA}.TEST_FEATURES")

feature_config = get_feature_config(config)
NUMERIC_COLS = feature_config["all_numeric_features"]
TARGET_COL = feature_config["target_column"]
OUTPUT_COL = "PREDICTED_RISK"


def calc_metrics(df, label_col, pred_col):
    return {
        "accuracy": accuracy_score(df=df, y_true_col_names=label_col, y_pred_col_names=pred_col),
        "f1_weighted": f1_score(
            df=df, y_true_col_names=label_col, y_pred_col_names=pred_col, average="weighted"
        ),
        "precision_weighted": precision_score(
            df=df, y_true_col_names=label_col, y_pred_col_names=pred_col, average="weighted"
        ),
        "recall_weighted": recall_score(
            df=df, y_true_col_names=label_col, y_pred_col_names=pred_col, average="weighted"
        ),
    }


model_v1 = LogisticRegression(
    input_cols=NUMERIC_COLS,
    label_cols=[TARGET_COL],
    output_cols=[OUTPUT_COL],
    max_iter=200,
    random_state=42,
)
model_v1.fit(train_spdf)

pred_v1 = model_v1.predict(test_spdf)
metrics_v1 = calc_metrics(pred_v1, TARGET_COL, OUTPUT_COL)

print("Model v1 trained (LogisticRegression max_iter=200):")
for k, v in metrics_v1.items():
    print(f"  {k}: {v:.4f}")

print(f"\nDataset: {ds_name}/{ds_version}")
print(f"Train: {train_spdf.count():,} | Test: {test_spdf.count():,}")

## 3 | Register Model v1 in the Model Registry

The `Registry.log_model()` call:

1. Serializes the LogisticRegression model
2. Stores it as a Snowflake Model object
3. Records the schema from `sample_input_data`
4. Captures **lineage** back to the source Dataset (when using a Snowpark DF)
5. Stores metrics alongside the model version

### Key Parameters

| Parameter | Purpose |
|-----------|---------|
| `model` | The trained Python model object |
| `model_name` | Name in the registry |
| `version_name` | Version identifier |
| `sample_input_data` | Schema inference + lineage capture (use Snowpark DF for Dataset lineage) |
| `metrics` | Evaluation metrics stored with the version |
| `task` | Model type (classification, regression, etc.) |
| `target_platforms` | Where the model can run (WAREHOUSE, SPCS) |
| `comment` | Human-readable description |

In [ ]:
from snowflake.ml.model.task import Task
from snowflake.ml.registry import Registry

MODEL_NAME = "PATIENT_RISK_MODEL_DEMO"

registry = Registry(
    session=session,
    database_name=DB,
    schema_name=SCHEMA,
)

mv_v1 = registry.log_model(
    model=model_v1,
    model_name=MODEL_NAME,
    version_name="v2",
    sample_input_data=train_spdf.select(NUMERIC_COLS).limit(10),
    metrics=metrics_v1,
    task=Task.TABULAR_MULTI_CLASSIFICATION,
    target_platforms=["SNOWPARK_CONTAINER_SERVICES"],
    comment=f"LogisticRegression max_iter=200 -- trained {datetime.now().isoformat()}",
)

print(f"Model registered: {MODEL_NAME}/v1")
print(f"Functions available: {[f['name'] for f in mv_v1.show_functions()]}")

In [ ]:
mv_v1.lineage(direction="upstream")

## 4 | Explore the Model Registry

The Model Registry provides a centralized catalog of all models in your schema.
You can list models, inspect versions, and query metadata -- all via SQL or Python.

In [ ]:
print("Models in registry:")
session.sql(f"SHOW MODELS IN SCHEMA {DB}.{SCHEMA}").show()

In [ ]:
model_ref = registry.get_model(MODEL_NAME)

print(f"Model: {MODEL_NAME}")
print(f"Versions: {[v.version_name for v in model_ref.versions()]}")
print()

print("Version v1 details:")
print(f"  Comment: {mv_v1.comment}")
print(f"  Functions: {[f['name'] for f in mv_v1.show_functions()]}")

## 5 | Log Additional Metrics to Model Version

After registration, you can log additional metrics to a model version.
This is useful for per-class metrics, custom business metrics, or metrics
computed on different evaluation datasets.

In [ ]:
class_labels = sorted(pred_v1.select(TARGET_COL).distinct().to_pandas()[TARGET_COL].tolist())
pred_pdf = pred_v1.select(TARGET_COL, OUTPUT_COL).to_pandas()

print("Logging per-class F1 scores:")
for label in class_labels:
    from sklearn.metrics import f1_score as sk_f1

    y_true_bin = (pred_pdf[TARGET_COL] == label).astype(int)
    y_pred_bin = (pred_pdf[OUTPUT_COL] == label).astype(int)
    f1_val = float(sk_f1(y_true_bin, y_pred_bin, zero_division=0))
    metric_name = f"f1_{label.lower()}"
    mv_v1.set_metric(metric_name, f1_val)
    print(f"  {metric_name}: {f1_val:.4f}")

mv_v1.set_metric("test_size", len(pred_pdf))
print(f"  test_size: {len(pred_pdf)}")

In [ ]:
print("All metrics for v1:")
all_metrics = mv_v1.show_metrics()
for k, v in all_metrics.items():
    print(f"  {k}: {v}")

## 6 | Model Lineage

Snowflake automatically captures the full provenance chain:

```
  RAW_PATIENT_DATA (source table)
       |
       v
  PATIENT_FEATURES/v1 (Feature View)
       |
       v
  PATIENT_RISK_TRAINING_DS (Dataset)
       |
       v
  PATIENT_RISK_MODEL_DEMO/v1 (Model)  <-- we are here
```

Lineage is captured automatically because we passed a **Snowpark DataFrame**
(from the Dataset) as `sample_input_data` to `log_model()`. The registry
traces back through the DataFrame's provenance to the source Dataset.

You can also view this visually in **Snowsight --> AI & ML --> Models --> Lineage tab**.

In [ ]:
print("Upstream lineage for model v1:")
print("(What data was used to create this model?)")
print()

upstream = mv_v1.lineage(direction="upstream")
for node in upstream:
    print(f"  {node._lineage_node_domain:15s} : {node._lineage_node_name}")

## 7 | Register a Second Model Version (v2)

Let's train a different configuration and register it as v2.
This demonstrates how the registry manages multiple versions,
enabling side-by-side comparison.

In [ ]:
model_v2 = LogisticRegression(
    input_cols=NUMERIC_COLS,
    label_cols=[TARGET_COL],
    output_cols=[OUTPUT_COL],
    max_iter=500,
    C=0.5,
    random_state=42,
)
model_v2.fit(train_spdf)

pred_v2 = model_v2.predict(test_spdf)
metrics_v2 = calc_metrics(pred_v2, TARGET_COL, OUTPUT_COL)

mv_v2 = registry.log_model(
    model=model_v2,
    model_name=MODEL_NAME,
    version_name="v2",
    sample_input_data=train_spdf.select(NUMERIC_COLS).limit(10),
    metrics=metrics_v2,
    task=Task.TABULAR_MULTI_CLASSIFICATION,
    target_platforms=["SNOWPARK_CONTAINER_SERVICES"],
    comment=f"LogisticRegression max_iter=500 C=0.5 -- trained {datetime.now().isoformat()}",
)

print(f"Model registered: {MODEL_NAME}/v2")
print("v2 metrics:")
for k, v in metrics_v2.items():
    print(f"  {k}: {v:.4f}")

## 8 | Compare Model Versions Side-by-Side

With multiple versions registered, we can compare them to decide which to promote.

In [ ]:
comparison = pd.DataFrame(
    {
        "v1 (LR max_iter=200)": metrics_v1,
        "v2 (LR max_iter=500 C=0.5)": metrics_v2,
    }
).T
comparison.index.name = "Version"

print("Model Version Comparison")
print("=" * 70)
display(comparison.round(4))
print()

for metric in comparison.columns:
    v1_val = comparison.loc["v1 (LR max_iter=200)", metric]
    v2_val = comparison.loc["v2 (LR max_iter=500 C=0.5)", metric]
    delta = v2_val - v1_val
    arrow = "^" if delta > 0 else "v" if delta < 0 else "="
    print(f"  {metric}: {arrow} {delta:+.4f}")

## 9 | Promotion Workflow

Before a model goes to production, it must meet minimum quality thresholds.
This is the **promotion gate** -- a critical step in responsible ML.

### Promotion Thresholds (from config)

| Metric | Threshold |
|--------|-----------|
| `accuracy` | >= 0.80 |
| `f1_weighted` | >= 0.75 |

### Promotion Decision Logic

```
  For each candidate version:
    1. Check all metrics against thresholds
    2. If ALL pass --> APPROVED for promotion
    3. If ANY fail --> REJECTED

  If approved, set as the default model version.
```

In [ ]:
THRESHOLDS = {
    "accuracy": config.evaluation.accuracy_threshold,
    "f1_weighted": config.evaluation.f1_macro_threshold,
}

print("Promotion Thresholds:")
for k, v in THRESHOLDS.items():
    print(f"  {k} >= {v}")
print()


def check_promotion(version_name, version_metrics):
    print(f"--- Checking {version_name} ---")
    all_passed = True
    for metric, threshold in THRESHOLDS.items():
        actual = version_metrics.get(metric, 0)
        passed = actual >= threshold
        status = "PASS" if passed else "FAIL"
        print(f"  {metric}: {actual:.4f} vs {threshold:.2f} --> {status}")
        if not passed:
            all_passed = False
    verdict = "APPROVED" if all_passed else "REJECTED"
    print(f"  Decision: {verdict}")
    return all_passed


v1_approved = check_promotion("v1", metrics_v1)
print()
v2_approved = check_promotion("v2", metrics_v2)

In [ ]:
best_version = None
best_f1 = 0

for vname, vmetrics, approved in [("v1", metrics_v1, v1_approved), ("v2", metrics_v2, v2_approved)]:
    if approved and vmetrics["f1_weighted"] > best_f1:
        best_version = vname
        best_f1 = vmetrics["f1_weighted"]

if best_version:
    model_ref = registry.get_model(MODEL_NAME)
    model_ref.default = best_version
    print(f"Default version set to: {best_version} (f1_weighted={best_f1:.4f})")
    print(f"This version will be used when no version is specified in inference calls.")
else:
    print("No version met promotion criteria. Default version unchanged.")

## 10 | Full Lifecycle Recap

Here is the complete lifecycle we built across all 4 notebooks:

```
  +-------------------+     +-------------------+     +-------------------+
  |  RAW_PATIENT_DATA |---->| PATIENT_FEATURES  |---->| Training Dataset  |
  |  (Source Table)   |     | (Feature View/DT) |     | (Immutable, PIT)  |
  +-------------------+     +-------------------+     +-------------------+
                                                              |
                                                              v
  +-------------------+     +-------------------+     +-------------------+
  |    (Future)       |<----| PATIENT_RISK_MODEL|<----| Experiment Runs   |
  | Deploy & Monitor  |     | (Model Registry)  |     | (4 variations)    |
  | (Sessions 3 & 4)  |     | v1, v2 + metrics  |     | Compare & select  |
  +-------------------+     +-------------------+     +-------------------+
```

### What We Covered in This Session

| Notebook | Topic | Key Snowflake Service |
|----------|-------|----------------------|
| 01 | Feature Engineering | Feature Store (Entity, FeatureView, Dynamic Tables) |
| 02 | Advanced Features | Datasets, Online Store, Lineage, RBAC Sharing |
| 03 | Experiment Tracking | ExperimentTracking API |
| 04 | Model Registry | Registry, Versioning, Promotion |

### What Comes Next (Future Sessions)

| Session | Topics |
|---------|--------|
| Session 3 | Distributed training, hyperparameter tuning |
| Session 4 | Model deployment (SPCS), inference, MLOps |

## 11 | Summary

In this notebook we:

1. **Trained** a LogisticRegression directly on Snowpark DataFrames using `snowflake.ml.modeling`
2. **Registered** the model as v1 in the Model Registry with Dataset lineage
3. **Explored** the registry -- listing models, versions, and metadata
4. **Logged** per-class metrics to the model version
5. **Traced** model lineage upstream to dataset, feature view, and source table
6. **Registered** a second version (v2) with different hyperparameters
7. **Compared** v1 vs v2 metrics side-by-side
8. **Ran** a promotion workflow with configurable thresholds
9. **Set** the best approved version as the default

### Key Takeaways

| Takeaway | Detail |
|----------|--------|
| **Snowflake ML modeling** | `snowflake.ml.modeling` models work directly with Snowpark DataFrames |
| **No preprocessing needed** | LogisticRegression handles string labels natively |
| **Automatic lineage** | Pass a Snowpark DF from a Dataset as `sample_input_data` to capture lineage |
| **One log_model() call** | Serializes, stores, captures schema, lineage, and metrics |
| **Promotion gates** | Configurable thresholds prevent bad models from reaching production |
| **Default version** | Clean abstraction for "which version to use in production" |

---

**End of Session 2.** Thank you for attending the deep dive!